# Multi‑Direction Depletion: Dimensionality of Typosquat Subspace
Iteratively projects out top‑k probe directions and measures AUC collapse.

In [ ]:
!pip install -q torch transformers scikit-learn matplotlib tqdm

In [ ]:
import json, random, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

import time, threading
def keep_alive():
    while True: time.sleep(60)
threading.Thread(target=keep_alive, daemon=True).start()

In [ ]:
DATASET_PATH = "typosquat_dataset_full/typosquat_tool_calls.jsonl"
df_all = pd.DataFrame([json.loads(line) for line in open(DATASET_PATH)])
df_adv = df_all[df_all['is_adversarial'] == True].sample(n=500, random_state=SEED).copy()
print(f"Sampled {len(df_adv)} adversarial entries")

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()

In [ ]:
def extract_fast(model, tokenizer, commands, batch_size=32):
    features = []
    for i in tqdm(range(0, len(commands), batch_size), desc="Extracting"):
        batch = commands[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=64).to(device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            hidden = outputs.hidden_states[14][:, -1, :].float().cpu().numpy()
        features.append(hidden)
    return np.vstack(features)

texts, labels = [], []
for _, row in df_adv.iterrows():
    texts.extend([row['clean_command'], row['typo_command']])
    labels.extend([0, 1])

X = extract_fast(model, tokenizer, texts)
y = np.array(labels)
print(f"Feature matrix: {X.shape}")

In [ ]:
def depletion_projection(X, directions):
    X_proj = X.copy()
    for w in directions:
        w_unit = w / np.linalg.norm(w)
        X_proj = X_proj - np.outer(X_proj @ w_unit, w_unit)
    return X_proj

max_directions = 30
auc_history = []
directions = []
X_current = X.copy()

for k in range(max_directions):
    probe = LogisticRegression(max_iter=2000, random_state=SEED)
    probe.fit(X_current, y)
    auc = roc_auc_score(y, probe.predict_proba(X_current)[:, 1])
    w = probe.coef_[0]
    directions.append(w)
    auc_history.append(auc)
    X_current = depletion_projection(X_current, [w])
    print(f"Direction {k+1}: AUC = {auc:.4f}")
    if auc < 0.6:
        print(f"Subspace collapsed after {k+1} directions.")
        break

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(range(1, len(auc_history)+1), auc_history, marker='o')
plt.axhline(y=0.5, linestyle='--', color='gray', label='Chance (0.5)')
plt.xlabel('Number of Depleted Directions')
plt.ylabel('Probe AUC')
plt.title('Multi‑Direction Depletion: Dimensionality of Typosquat Subspace')
plt.grid(alpha=0.3)
plt.legend()
plt.savefig('multi_direction_depletion.png', dpi=150)
plt.show()

print(f"Subspace dimensionality (AUC > 0.6): {len(auc_history)}")